In [1]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [3]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
    """The Grand Canyon is one of the most visited natural wonders in the world.
    Photosynthesis is the process by which green plants convert sunlight into energy.
    Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
    """In medieval Europe, castles were built primarily for defense.
    The chlorophyll in plant cells captures sunlight during photosynthesis.
    Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
    """Basketball was invented by Dr. James Naismith in the late 19th century.
    It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
    """The history of cinema began in the late 1800s. Silent films were the earliest form.
    Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
    Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]


In [7]:
# Defining embedding model and creating vector store

embedding_model  = OpenAIEmbeddings()

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [14]:
# Creating similarity search retriever
base_retriever = vector_store.as_retriever(kwargs={'k':2})

In [15]:
# Creating Contextual Compression Retriever
# To create the CCR object, ContextualCompressionRetriever() Class is used.
# Parameters:
#   base_retriever - Provide a basic retriever like similarity search or mmr
#   base_compressor - We use the function from_llm() from LLMChainExtractor to create a compressor
#       Here we have provide  Chat model in order to refine the output result using LLM.

ccr = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=LLMChainExtractor.from_llm(llm=ChatOpenAI(model='gpt-5-nano'))
)

In [16]:
# Providing a query
query = 'What is photosynthesis'

In [20]:
# Generating outputs based on normal retriever, just for comparision
normal_result = base_retriever.invoke(query)
[print(doc.page_content) for doc in normal_result]

# Here we can see some irrelevant resultbeing provided by the normal retriever.
# Reason - Because the document fetch by the retriever contains the extra unnecessary details

The Grand Canyon is one of the most visited natural wonders in the world.
    Photosynthesis is the process by which green plants convert sunlight into energy.
    Millions of tourists travel to see it every year. The rocks date back millions of years.
In medieval Europe, castles were built primarily for defense.
    The chlorophyll in plant cells captures sunlight during photosynthesis.
    Knights wore armor made of metal. Siege weapons were often used to breach castle walls.
The history of cinema began in the late 1800s. Silent films were the earliest form.
    Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
    Modern filmmaking involves complex CGI and sound design.
Basketball was invented by Dr. James Naismith in the late 19th century.
    It was originally played with a soccer ball and peach baskets. NBA is now a global league.


[None, None, None, None]

In [ ]:
# Generating compressed results using Contextual Compression Retriever
compressed_results = ccr.invoke(query)
[print(doc.page_content) for doc in compressed_results]

# Here the results are refined and trimmed to provide only relevant required details

Photosynthesis is the process by which green plants convert sunlight into energy.
The chlorophyll in plant cells captures sunlight during photosynthesis.
Photosynthesis does not occur in animal cells.


[None, None, None]